# Notebook 03 — Feature Engineering and Targets

**FIN 4600 · Lab 3 · Financial Data Analytics**

Duran, *Financial Services Technology* (3rd ed.), **Chapter 6 — Data Analytics**

---

This is the bridge between descriptive and predictive analytics, and it is
where most of the real work — and most of the real mistakes — live.

A machine learning model needs a rectangular table: one row per observation,
one column per **feature**, plus a **target** column holding the thing you are
trying to predict. Building that table from a time series is not hard. Building
it *without accidentally leaking the future into it* is hard, and it is the
difference between a model that works and a backtest that lies to you.

**What you will learn**

1. Building per-instrument features with `groupby().transform()`
2. Joining market-wide data (VIX) onto per-instrument data
3. The lag discipline — what you knew, and when you knew it
4. Look-ahead bias, demonstrated rather than described
5. Defining a target without leaking it into the features
6. Why a random train/test split is wrong for time series
7. Scaling, and why it is fitted on training data only

## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root(start=None):
    """Return the repository root — the folder that contains data/sp500_prices.csv."""
    here = Path(start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "sp500_prices.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the repository root. In VS Code use File > Open Folder "
        "and open the mtu4600-lab3-analytics folder itself, then re-run."
    )


REPO = find_repo_root()
DATA = REPO / "data"
plt.style.use(REPO / "fin4600.mplstyle")

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 150)

In [ ]:
prices = pd.read_csv(DATA / "sp500_prices.csv", parse_dates=["date"])
companies = pd.read_csv(DATA / "sp500_companies.csv")
vix = pd.read_csv(DATA / "vix_daily.csv", parse_dates=["date"])

prices = prices.sort_values(["ticker", "date"]).reset_index(drop=True)
print(f"{len(prices):,} rows, {prices['ticker'].nunique()} tickers")
prices.head(3)

## 1. Per-instrument features with `groupby().transform()`

Every feature here must be computed **within** a ticker. If you compute a
21-day return on the whole sorted frame, the first row of MSFT will be
compared with the last row of MMM, which is nonsense.

`groupby("ticker").transform(...)` applies a function inside each group and
returns a result aligned to the original rows. It is the single most useful
pattern for panel data.

In [ ]:
grouped = prices.groupby("ticker")["close"]

# Trailing returns over several horizons — "momentum"
for days, name in [(21, "1m"), (63, "3m"), (126, "6m"), (252, "12m")]:
    prices[f"momentum_{name}"] = grouped.transform(lambda s, d=days: s / s.shift(d) - 1)

# Trailing realised volatility, annualised
daily_return = grouped.transform(lambda s: s.pct_change())
prices["daily_return"] = daily_return
prices["volatility_21d"] = (
    prices.groupby("ticker")["daily_return"].transform(lambda s: s.rolling(21).std())
    * np.sqrt(252)
)
prices["volatility_63d"] = (
    prices.groupby("ticker")["daily_return"].transform(lambda s: s.rolling(63).std())
    * np.sqrt(252)
)

# Where is the price relative to its own trailing average and its 52-week high?
prices["px_vs_200d_ma"] = grouped.transform(lambda s: s / s.rolling(200).mean() - 1)
prices["px_vs_52w_high"] = grouped.transform(lambda s: s / s.rolling(252).max() - 1)

# Is volume unusual right now, measured in its own standard deviations?
log_volume = np.log(prices["volume"])
prices["volume_zscore"] = prices.assign(lv=log_volume).groupby("ticker")["lv"].transform(
    lambda s: (s - s.rolling(63).mean()) / s.rolling(63).std()
)

feature_columns = [
    "momentum_1m",
    "momentum_3m",
    "momentum_6m",
    "momentum_12m",
    "volatility_21d",
    "volatility_63d",
    "px_vs_200d_ma",
    "px_vs_52w_high",
    "volume_zscore",
]

(
    prices.loc[prices["ticker"] == "AAPL", ["date", "close"] + feature_columns]
    .tail(3)
    .assign(date=lambda d: d["date"].dt.date)
    .round(4)
)

### Sanity check: did the grouping actually work?

If the features had been computed across the whole frame instead of within
each ticker, the first row of each ticker after the first would carry a
nonsense value instead of `NaN`. Check it.

In [ ]:
first_rows = prices.groupby("ticker").head(1)
print("Non-null feature values on each ticker's FIRST row (should all be 0):")
print(first_rows[feature_columns].notna().sum().to_string())

## 2. Joining market-wide data

The VIX is one number per day, shared by every instrument. A `merge` on
`date` broadcasts it across all 30 tickers for that day.

In [ ]:
vix_features = vix[["date", "vix_close"]].copy()
vix_features["vix_change_5d"] = vix_features["vix_close"].pct_change(5)
vix_features["vix_vs_63d_avg"] = (
    vix_features["vix_close"] / vix_features["vix_close"].rolling(63).mean() - 1
)

before = len(prices)
panel = prices.merge(vix_features, on="date", how="left")
print(f"Rows before merge: {before:,}   after: {len(panel):,}   (must match)")
print(f"Rows with no VIX value: {panel['vix_close'].isna().sum()}")

feature_columns += ["vix_close", "vix_change_5d", "vix_vs_63d_avg"]

In [ ]:
# Attach sector for later analysis (not used as a model feature here)
panel = panel.merge(companies[["ticker", "sector"]], on="ticker", how="left")
print("Unmatched tickers:", panel.loc[panel["sector"].isna(), "ticker"].unique().tolist() or "none")

## 3. The lag discipline

This is the heart of the notebook.

Ask of every feature: **at the moment I would act on this, would I actually
have known it?**

Our features are built from rolling windows that end at the current row, so
the value in row $t$ uses prices up to and including the close on day $t$.
That is fine *if* the decision is made after the close on day $t$ and the
outcome starts on day $t+1$. It is not fine if you compare it with day $t$'s
own return — you would be using the closing price to predict itself.

The safe habit: **shift the features forward by one period**, so that row $t$
holds only information available strictly before $t$.

In [ ]:
lagged = panel.copy()
for column in feature_columns:
    lagged[column] = lagged.groupby("ticker")[column].shift(1)

comparison = pd.DataFrame(
    {
        "close": panel.loc[panel["ticker"] == "AAPL", "close"].to_numpy()[200:205],
        "momentum_1m (as computed)": panel.loc[panel["ticker"] == "AAPL", "momentum_1m"].to_numpy()[200:205],
        "momentum_1m (lagged)": lagged.loc[lagged["ticker"] == "AAPL", "momentum_1m"].to_numpy()[200:205],
    }
)
comparison.round(4)

The lagged column is the same series moved down one row. That one-row shift is
often all that separates a defensible model from a fraudulent one.

## 4. Look-ahead bias, demonstrated

Rather than take this on faith, build a deliberately cheating feature and see
what it does. A **centred** rolling window is the classic accident — it looks
innocuous and it uses future data.

In [ ]:
# Honest: a trailing 5-day average, ending today
panel["ma5_trailing"] = panel.groupby("ticker")["close"].transform(
    lambda s: s.rolling(5).mean()
)

# Cheating: a centred 5-day average, which peeks two days into the future
panel["ma5_centred"] = panel.groupby("ticker")["close"].transform(
    lambda s: s.rolling(5, center=True).mean()
)

# What we want to predict: tomorrow's return
panel["next_day_return"] = panel.groupby("ticker")["close"].transform(
    lambda s: s.shift(-1) / s - 1
)

check = panel.dropna(subset=["ma5_trailing", "ma5_centred", "next_day_return"])
honest_signal = check["close"] / check["ma5_trailing"] - 1
cheating_signal = check["close"] / check["ma5_centred"] - 1

print("Correlation with tomorrow's return:")
print(f"  Trailing 5-day average (honest):  {honest_signal.corr(check['next_day_return']):+.4f}")
print(f"  Centred 5-day average (cheating): {cheating_signal.corr(check['next_day_return']):+.4f}")

The honest signal has essentially no predictive correlation, which is what
five decades of market efficiency research would lead you to expect. The
cheating signal has a large one — and it is entirely an artefact of
`center=True` letting tomorrow's price into today's average.

**If a financial model suddenly starts working well, your first hypothesis
should be a leak, not a discovery.** Go and look for it before you tell
anyone.

In [ ]:
# Drop the cheating column so nobody uses it by accident
panel = panel.drop(columns=["ma5_centred"])

### Exercise 1

Here are three candidate features. For each, decide whether it leaks the
future, and explain why in a sentence:

1. `close / close.rolling(60).max() - 1` — distance below the 60-day high
2. `close / close.mean() - 1` — distance from the average price over the
   whole sample
3. `volume / volume.rolling(20).mean()` — today's volume relative to recent
   volume

Write your answers in the markdown cell below.

**Your answers:**

1.
2.
3.

## 5. From daily to monthly, and defining the target

Daily returns are mostly noise. Monthly observations give a model a better
shot and cut the sample to a size that trains in seconds, which matters in a
lab.

We take the **last observation of each month** for the features, and define
the target as the return of the **following** month.

In [ ]:
panel = panel.sort_values(["ticker", "date"])
panel["month"] = panel["date"].dt.to_period("M")

monthly = (
    panel.groupby(["ticker", "month"])
    .last()
    .reset_index()
    .sort_values(["ticker", "month"])
)

print(f"Daily panel: {len(panel):,} rows")
print(f"Monthly panel: {len(monthly):,} rows "
      f"({monthly['ticker'].nunique()} tickers x ~{len(monthly) // monthly['ticker'].nunique()} months)")

In [ ]:
# Target: the return of the NEXT month, computed from month-end closes
monthly["next_month_return"] = monthly.groupby("ticker")["close"].transform(
    lambda s: s.shift(-1) / s - 1
)
monthly["next_month_up"] = (monthly["next_month_return"] > 0).astype("Int64")
monthly.loc[monthly["next_month_return"].isna(), "next_month_up"] = pd.NA

# A second, different target: next month's realised volatility
monthly["next_month_volatility"] = monthly.groupby("ticker")["volatility_21d"].shift(-1)

(
    monthly[["ticker", "month", "close", "momentum_3m", "next_month_return", "next_month_up"]]
    .head(15)
    .astype({"month": str})
    .round(4)
)

### Check the target for leakage

The target must not be derivable from any feature. The quickest check is a
correlation scan: a feature correlating unusually strongly with the target is
either a genuine discovery or, far more likely, a leak.

In [ ]:
model_data = monthly.dropna(subset=feature_columns + ["next_month_return"]).copy()

correlations = (
    model_data[feature_columns]
    .corrwith(model_data["next_month_return"])
    .sort_values(key=abs, ascending=False)
)
print("Correlation of each feature with next month's return:")
print(correlations.round(4).to_string())
print(f"\nLargest absolute correlation: {correlations.abs().max():.4f}")

Everything is small. That is the correct and slightly deflating result: these
features carry very little information about next month's direction. A
correlation of 0.6 here would have meant a bug, not a trading strategy.

Notebook 05 takes this honestly — and then shows a target where the same
features *do* work.

In [ ]:
print(f"Rows available for modelling: {len(model_data):,}")
print(f"Date range: {model_data['month'].min()} to {model_data['month'].max()}")
print(f"\nClass balance of next_month_up:")
print(model_data["next_month_up"].value_counts(normalize=True).round(3).to_string())

Roughly 57/43 in favour of "up". That base rate is the number any classifier
must beat. A model that always predicts "up" is 57% accurate and completely
useless — which is why accuracy alone is a bad metric, a point Notebook 04
develops.

## 6. Splitting the data — by time, never at random

For cross-sectional data (like the loan applications in Notebook 04) a random
split is right. For time series it is wrong, for two reasons:

1. **Temporal leakage.** If March 2015 is in training and February 2015 is in
   testing, the model has seen the future relative to its test set.
2. **Cross-sectional leakage.** Our panel has 30 tickers on the same date.
   A random split puts some of a given month's rows in training and the rest
   in testing. Since stocks are correlated within a month, the model can
   effectively read the answer off its training rows.

Split on a date. Everything before the cut trains, everything after tests.

In [ ]:
SPLIT_DATE = pd.Period("2016-06", freq="M")

train = model_data[model_data["month"] < SPLIT_DATE]
test = model_data[model_data["month"] >= SPLIT_DATE]

print(f"Train: {len(train):>5,} rows   {train['month'].min()} to {train['month'].max()}")
print(f"Test:  {len(test):>5,} rows   {test['month'].min()} to {test['month'].max()}")
print(f"\nNo month appears in both sets: {set(train['month']).isdisjoint(set(test['month']))}")

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 2.6))
train_months = sorted(train["month"].unique())
test_months = sorted(test["month"].unique())
ax.barh([0], [len(train_months)], left=[0], color="#2a78d6", height=0.5, label="Train")
ax.barh([0], [len(test_months)], left=[len(train_months)], color="#eb6834", height=0.5, label="Test")
ax.set_yticks([])
ax.set_xlabel("Months since start of sample")
ax.set_title("A time-series split: train on the past, test on the future")
ax.grid(False)
ax.legend(loc="lower right", ncols=2)
ax.text(len(train_months) / 2, 0, f"{len(train_months)} months", ha="center", va="center",
        color="white", fontsize=9, fontweight="bold")
ax.text(len(train_months) + len(test_months) / 2, 0, f"{len(test_months)} months",
        ha="center", va="center", color="white", fontsize=9, fontweight="bold")
plt.show()

## 7. Scaling — fit on train only

Many models need features on comparable scales. `volatility_21d` runs around
0.2 while `vix_close` runs around 15; a distance-based or regularised model
will let the VIX dominate purely because of its units.

The rule that students get wrong: **fit the scaler on the training data
only**, then apply it to the test data. Fitting on everything lets the test
set's mean and standard deviation — information from the future — influence
the training features. It is a small leak, but it is a leak, and graders
(and regulators) look for it.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_scaled = scaler.fit_transform(train[feature_columns])   # fit AND transform
test_scaled = scaler.transform(test[feature_columns])         # transform ONLY

print("Training features after scaling — mean ≈ 0, SD ≈ 1 by construction:")
print(pd.DataFrame(train_scaled, columns=feature_columns).describe().loc[["mean", "std"]].round(3))

print("\nTest features after applying the SAME scaler — NOT exactly 0 and 1,")
print("and that is correct. The test set is a different period.")
print(pd.DataFrame(test_scaled, columns=feature_columns).describe().loc[["mean", "std"]].round(3))

In Notebook 04 we stop doing this by hand and use a scikit-learn `Pipeline`,
which makes the leak structurally impossible rather than merely discouraged.

### Exercise 2

Add one feature of your own to `feature_columns` and rebuild `model_data`.
Some ideas:

- the spread between 21-day and 63-day volatility (a term structure of risk)
- the ratio of `momentum_1m` to `momentum_12m` (short-term reversal against
  long-term trend)
- a calendar feature such as the month number, to test the "January effect"

State in one sentence, before you compute anything, whether your feature
could leak the future. Then check its correlation with `next_month_return`.

In [ ]:
# YOUR CODE HERE

## 8. Save the modelling table

In [ ]:
processed = DATA / "processed"
processed.mkdir(exist_ok=True)

save_columns = (
    ["ticker", "month", "date", "sector", "close"]
    + feature_columns
    + ["next_month_return", "next_month_up", "next_month_volatility"]
)
model_data[save_columns].assign(month=lambda d: d["month"].astype(str)).to_csv(
    processed / "monthly_features.csv", index=False
)

print("Wrote", processed / "monthly_features.csv")
print(f"{len(model_data):,} rows x {len(save_columns)} columns")
print("\nFeatures:", ", ".join(feature_columns))

---

## Recap

1. `groupby().transform()` computes features within each instrument.
2. After every merge, check the row count.
3. Ask of every feature: would I have known this at the time?
4. A centred rolling window is a leak. So is any statistic computed over the
   whole sample.
5. A model that suddenly works is a leak until proven otherwise.
6. Split time series by date. Random splits leak, twice over.
7. Fit scalers on training data only.

Next: `04_ml_credit_risk.ipynb` — supervised learning on a problem where the
signal is real and strong, so you can learn the machinery before facing the
market's noise.